# Diamond Listings Data Collection
**Beyond the 4Cs: An Analysis of Provenance, Production Method, and the Distinct Structure of Diamond Pricing**
*Boemo Seboni | DSS 660 – Summer 2026*

This notebook collects the dataset used in the project: **187,579 diamond listings** (94,448 natural and 93,131 lab-grown) from Brilliant Earth, a U.S. online diamond retailer, through the product-listing API that powers its website.

Natural diamonds are collected separately for each origin Brilliant Earth reports (**Botswana, Angola, Canada, and Recycled**) so that provenance can be tested as a pricing factor. Lab-grown diamonds are collected in a separate pass. The two sets are then combined into one file that was used for the statistical analysis in JMP.

**Workflow**
1. Confirm the API parameters and available fields
2. Test how far the API can paginate
3. Size the carat-weight bands for each origin
4. Collect natural diamonds by origin
5. Design carat-weight bands for lab-grown diamonds
6. Collect lab-grown diamonds
7. Combine both into the final dataset

**Output files:** `diamonds_provenance.csv` (natural), `diamonds_labgrown.csv` (lab-grown), `diamonds_merged_complete.csv` (combined)

> **Note:** The outputs below are from the original run (June 2026). Collecting the full dataset takes several hours, and live listings change daily, so re-running will produce slightly different counts. The raw data is not included in this repository because it consists of Brilliant Earth's product listings.

*Code developed with AI assistance.*

## 1. Confirm API parameters and available fields
Sends one small request for each natural-diamond origin to confirm the correct filter values (`origins`, `product_class`) and to list every field returned for a diamond.

In [4]:
import requests
import json

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
    "Accept": "application/json, text/plain, */*",
    "Referer": "https://www.brilliantearth.com/diamond/shop-all/",
}

BASE = "https://www.brilliantearth.com/api/v1/plp/products/"

# Test each origin value with the correct parameter name "origins"
origin_values = [
    "Botswana Sort",
    "Canada",
    "Angola",
    "Recycled",
    "",   # no origin filter = all natural
]

for origin in origin_values:
    params = {
        "display": 50,
        "page": 1,
        "currency": "USD",
        "product_class": "Loose Diamonds",
        "order_by": "most_popular",
        "order_method": "asc",
        "min_carat": 0.25,
        "max_carat": 20.0,
    }
    if origin:
        params["origins"] = origin

    r = requests.get(BASE, params=params, headers=headers)
    data = r.json()

    # Try to find total count and products
    total = (data.get("total") or
             data.get("data", {}).get("total") or
             data.get("count") or "?")

    products = (data.get("results") or
                data.get("data", {}).get("products") or
                data.get("products") or [])

    if products:
        first = products[0]
        # Show all keys from the first successful response to confirm the available fields
        if origin == "Botswana Sort":
            print(f"\n--- FIELD NAMES (from Botswana Sort result) ---")
            for k, v in first.items():
                print(f"  {k}: {repr(str(v))[:60]}")
        
        print(f"\norigins='{origin}' → total={total}, sample price=${first.get('price','?')}, "
              f"carat={first.get('carat','?')}, cut={first.get('cut','?')}, "
              f"color={first.get('color','?')}, clarity={first.get('clarity','?')}, "
              f"origin_field={first.get('origins', first.get('origin', 'NOT FOUND'))}")
    else:
        print(f"\norigins='{origin}' → total={total}, NO PRODUCTS. Keys: {list(data.keys())}")


--- FIELD NAMES (from Botswana Sort result) ---
  id: '76878064'
  upc: '18482630Y'
  title: '0.32 Carat Emerald Diamond'
  product_class: 'Loose Diamonds'
  price: '250'
  shipping_day: '5'
  shape: 'Emerald'
  color: 'J'
  clarity: 'VS1'
  cut: 'Ideal'
  report: 'GIA'
  origin: 'Botswana Sort'
  polish: 'Excellent'
  symmetry: 'Very Good'
  measurements: '4.69 x 3.19 x 2.18'
  girdle: 'Slightly Thick - Thick'
  culet: 'None'
  fluorescence: 'Faint'
  certificate_number: '1549301682'
  v360_src: '//css.brilliantearth.com/static/v360_viewer_zoom_slider/Vis
  diamond_collection: 'None'
  collection: ''
  receive_by: 'Friday, June 19'
  is_memo: 'False'
  has_cert: 'False'
  dedicated: 'False'
  diamond_dor: 'False'
  real_diamond_image: 'True'
  has_v360_video: 'True'
  diamond_blockchain: 'False'
  popular_choice: 'False'
  show_quick_ship_badge: 'True'
  hearts_and_arrows_diamonds: 'False'
  scs_certified: 'None'
  renewable_energy: 'None'
  only_one_available: 'False'
  truly_brilli

## 2. Test pagination limits
The API returns 50 diamonds per page. This test requests pages at increasing depths for Botswana diamonds to see where results stop. Because results stop well before the end of the full inventory, the collection is split into narrow carat-weight bands, each small enough to page through completely.

In [6]:
import requests
import time
import random

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
    "Accept": "application/json, text/plain, */*",
    "Referer": "https://www.brilliantearth.com/diamond/shop-all/",
}

BASE = "https://www.brilliantearth.com/api/v1/plp/products/"

# Test Botswana Sort specifically — scrape pages 1, 2, 3,
# then jump to page 50, 100, 200, 500, 1000, 1300
# This shows where (if anywhere) the API stops returning results

test_pages = [1, 2, 3, 10, 50, 100, 200, 500, 1000, 1100, 1200, 1300, 1310]

print("Pagination stress test — Botswana Sort")
print("="*55)

for page in test_pages:
    params = {
        "display": 50,
        "page": page,
        "currency": "USD",
        "product_class": "Loose Diamonds",
        "order_by": "most_popular",
        "order_method": "asc",
        "min_carat": 0.25,
        "max_carat": 20.0,
        "origins": "Botswana Sort",
    }

    try:
        r = requests.get(BASE, params=params, headers=headers, timeout=15)
        data = r.json()

        products = (data.get("results") or
                    data.get("data", {}).get("products") or
                    data.get("products") or [])

        total = (data.get("total") or
                 data.get("data", {}).get("total") or "?")

        # Check for any error messages in the response
        error = data.get("message") or data.get("error") or ""

        if products:
            first_id = products[0].get("id", "?")
            first_price = products[0].get("price", "?")
            print(f"  Page {page:>5}: ✅ {len(products)} products | "
                  f"total={total} | first_id={first_id} | price=${first_price}")
        else:
            print(f"  Page {page:>5}: ❌ 0 products | "
                  f"total={total} | error='{error}' | keys={list(data.keys())}")

    except Exception as e:
        print(f"  Page {page:>5}: 💥 EXCEPTION — {e}")

    time.sleep(random.uniform(1.5, 2.5))

print("\nTest complete.")
print("If all pages returned products → full scrape is safe to run")
print("If it breaks at a specific page → we need the carat band strategy")

Pagination stress test — Botswana Sort
  Page     1: ✅ 50 products | total=65503 | first_id=76878064 | price=$250
  Page     2: ✅ 50 products | total=65503 | first_id=80688488 | price=$300
  Page     3: ✅ 50 products | total=65503 | first_id=78820692 | price=$310
  Page    10: ✅ 50 products | total=65503 | first_id=75725825 | price=$360
  Page    50: ✅ 50 products | total=65503 | first_id=80691451 | price=$440
  Page   100: ✅ 50 products | total=65503 | first_id=80698540 | price=$500
  Page   200: ✅ 50 products | total=65503 | first_id=77427340 | price=$610
  Page   500: ✅ 50 products | total=65503 | first_id=77186980 | price=$610
  Page  1000: ✅ 50 products | total=65503 | first_id=77427340 | price=$610
  Page  1100: ✅ 50 products | total=65503 | first_id=77186980 | price=$610
  Page  1200: ✅ 50 products | total=65503 | first_id=77186980 | price=$610
  Page  1300: ✅ 50 products | total=65503 | first_id=77186980 | price=$610
  Page  1310: ✅ 50 products | total=65503 | first_id=77186980

## 3. Size the carat-weight bands for each origin
Counts how many diamonds fall in each carat band for each origin, and how many pages each band would need, to confirm every band can be collected in full.

In [7]:
import requests
import time

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
    "Accept": "application/json, text/plain, */*",
    "Referer": "https://www.brilliantearth.com/diamond/shop-all/",
}

BASE = "https://www.brilliantearth.com/api/v1/plp/products/"

# Carat bands reused from an earlier Brilliant Earth data collection
# Test each one for EACH origin to see how many diamonds fall in each band
carat_bands = [
    (0.25, 0.50),
    (0.51, 0.75),
    (0.76, 1.00),
    (1.01, 1.50),
    (1.51, 2.00),
    (2.01, 3.00),
    (3.01, 5.00),
    (5.01, 20.0),
]

origins = [
    ("Botswana Sort", "Botswana Sort"),
    ("Canada",        "Canada"),
    ("Angola",        "Angola"),
    ("No Filter",     ""),
]

print("Band sizing test — diamonds per carat band per origin")
print("="*75)
print(f"{'Origin':<20} {'Band':<15} {'Total':>10} {'Pages needed':>15} {'OK?':>6}")
print("-"*75)

for origin_label, origin_param in origins:
    band_totals = []
    for lo, hi in carat_bands:
        params = {
            "display": 50,
            "page": 1,
            "currency": "USD",
            "product_class": "Loose Diamonds",
            "order_by": "most_popular",
            "order_method": "asc",
            "min_carat": lo,
            "max_carat": hi,
        }
        if origin_param:
            params["origins"] = origin_param

        try:
            r = requests.get(BASE, params=params, headers=headers, timeout=15)
            data = r.json()
            total = (data.get("total") or
                     data.get("data", {}).get("total") or 0)
            pages_needed = -(-int(total) // 50)  # ceiling division
            ok = "✅" if pages_needed <= 450 else "⚠️ TOO BIG"
            band_totals.append(total)
            print(f"  {origin_label:<18} {str(lo)+'-'+str(hi):<15} {total:>10,} {pages_needed:>15,} {ok:>6}")
        except Exception as e:
            print(f"  {origin_label:<18} {str(lo)+'-'+str(hi):<15} ERROR: {e}")

        time.sleep(1.2)

    print(f"  {'TOTAL '+origin_label:<18} {'ALL':<15} {sum(band_totals):>10,}")
    print()

print("Any band marked ⚠️ TOO BIG needs to be split further.")
print("All ✅ = ready to scrape.")

Band sizing test — diamonds per carat band per origin
Origin               Band                 Total    Pages needed    OK?
---------------------------------------------------------------------------
  Botswana Sort      0.25-0.5            17,542             351      ✅
  Botswana Sort      0.51-0.75           14,454             290      ✅
  Botswana Sort      0.76-1.0             9,653             194      ✅
  Botswana Sort      1.01-1.5            14,039             281      ✅
  Botswana Sort      1.51-2.0             4,908              99      ✅
  Botswana Sort      2.01-3.0             3,432              69      ✅
  Botswana Sort      3.01-5.0             1,171              24      ✅
  Botswana Sort      5.01-20.0              304               7      ✅
  TOTAL Botswana Sort ALL                 65,503

  Canada             0.25-0.5             2,777              56      ✅
  Canada             0.51-0.75            2,699              54      ✅
  Canada             0.76-1.0          

## 4. Collect natural diamonds by origin
Collects every listing in each carat band for each origin (Recycled, Canada, Angola, then Botswana), saving a checkpoint file after each origin so progress isn't lost.

This first run collected **Recycled** and **Canada** in full, then was stopped manually after a network outage interrupted the Angola collection (the error at the end of the output).

In [14]:
import requests
import pandas as pd
import time
import random


headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
    "Accept": "application/json, text/plain, */*",
    "Referer": "https://www.brilliantearth.com/diamond/shop-all/",
}

BASE = "https://www.brilliantearth.com/api/v1/plp/products/"

FIELDS = [
    "id", "price", "carat", "cut", "color", "clarity", "shape",
    "origin", "report", "polish", "symmetry", "fluorescence",
    "depth", "table", "length_width_ratio", "length",
]

CARAT_BANDS = [
    (0.25, 0.50),
    (0.51, 0.75),
    (0.76, 1.00),
    (1.01, 1.50),
    (1.51, 2.00),
    (2.01, 3.00),
    (3.01, 5.00),
    (5.01, 20.0),
]

CHECKPOINT_PATH = "diamonds_provenance_checkpoint.csv"
OUTPUT_PATH     = "diamonds_provenance.csv"

def scrape_band(origin_label, origin_param, carat_min, carat_max):
    records = []
    page = 1
    consecutive_errors = 0
    total_available = None
    seen_ids = set()

    while True:
        params = {
            "display": 50,
            "page": page,
            "currency": "USD",
            "product_class": "Loose Diamonds",
            "order_by": "most_popular",
            "order_method": "asc",
            "min_carat": carat_min,
            "max_carat": carat_max,
        }
        if origin_param:
            params["origins"] = origin_param

        try:
            r = requests.get(BASE, params=params, headers=headers, timeout=15)

            if r.status_code != 200:
                consecutive_errors += 1
                print(f"      HTTP {r.status_code} on page {page} — waiting 15s")
                if consecutive_errors >= 5:
                    print(f"      5 errors — stopping band {carat_min}-{carat_max}")
                    break
                time.sleep(15)
                continue

            data = r.json()
            products = (data.get("results") or
                        data.get("data", {}).get("products") or
                        data.get("products") or [])

            if total_available is None:
                total_available = int(data.get("total") or
                                      data.get("data", {}).get("total") or 0)

            if not products:
                break

            # Duplicate detection — stop if API is looping
            page_ids = {p.get("id") for p in products}
            if page_ids and page_ids.issubset(seen_ids):
                print(f"      Page {page}: duplicate loop detected — stopping band")
                break
            seen_ids.update(page_ids)

            for p in products:
                rec = {field: p.get(field) for field in FIELDS}
                rec["origin_group"] = origin_label
                records.append(rec)

            consecutive_errors = 0

            if page * 50 >= total_available:
                break

            page += 1
            time.sleep(random.uniform(0.8, 1.8))

        except Exception as e:
            consecutive_errors += 1
            print(f"      Exception page {page}: {e}")
            if consecutive_errors >= 5:
                break
            time.sleep(10)

    return records


def scrape_origin(origin_label, origin_param):
    all_records = []
    print(f"\n{'='*60}")
    print(f"ORIGIN: {origin_label}")
    print(f"{'='*60}")

    for carat_min, carat_max in CARAT_BANDS:
        print(f"  Band {carat_min}-{carat_max}ct...", end=" ", flush=True)
        band_records = scrape_band(origin_label, origin_param, carat_min, carat_max)
        all_records.extend(band_records)
        print(f"{len(band_records):,} records | Running total: {len(all_records):,}")
        time.sleep(random.uniform(2, 4))

    print(f"ORIGIN DONE: {len(all_records):,} total for {origin_label}")
    return all_records


# ── SCRAPE ────────────────────────────────────────────────────────────────────
# Estimated runtimes:
#   Recycled:      ~1 min  (143 diamonds)
#   Canada:        ~10 min (10,182 diamonds)
#   Angola:        ~30 min (35,718 diamonds)
#   Botswana Sort: ~55 min (65,503 diamonds)
#   TOTAL:         ~100 min

scrape_targets = [
    ("Recycled",      "Recycled"),
    ("Canada",        "Canada"),
    ("Angola",        "Angola"),
    ("Botswana Sort", "Botswana Sort"),
]

all_data = []

for label, param in scrape_targets:
    records = scrape_origin(label, param)
    all_data.extend(records)

    # Checkpoint save after every origin
    pd.DataFrame(all_data).to_csv(CHECKPOINT_PATH, index=False)
    print(f"Checkpoint saved: {len(all_data):,} total rows → {CHECKPOINT_PATH}")
    time.sleep(random.uniform(5, 10))


# ── CLEAN & SAVE ──────────────────────────────────────────────────────────────
df = pd.DataFrame(all_data)

# Deduplicate by diamond ID (boundary carats can appear in two bands)
before = len(df)
df = df.drop_duplicates(subset=["id"])
print(f"\nDeduplication: {before:,} → {len(df):,} rows ({before - len(df):,} dupes removed)")

# Type conversions
for col in ["price", "carat", "depth", "table", "length_width_ratio", "length"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# Drop rows missing price or carat
df = df.dropna(subset=["price", "carat"])

df.to_csv(OUTPUT_PATH, index=False)

# ── SUMMARY ───────────────────────────────────────────────────────────────────
print(f"\n{'='*60}")
print(f"FINAL DATASET SAVED: diamonds_provenance.csv")
print(f"{'='*60}")
print(f"Total rows: {len(df):,}")

print(f"\nCount by origin_group:")
print(df["origin_group"].value_counts().to_string())

print(f"\nCount by raw origin field:")
print(df["origin"].value_counts().to_string())

print(f"\nMedian price by origin:")
print(df.groupby("origin_group")["price"].median().sort_values(ascending=False).to_string())

print(f"\nMedian carat by origin:")
print(df.groupby("origin_group")["carat"].median().sort_values(ascending=False).to_string())

print(f"\nCut distribution by origin:")
print(pd.crosstab(df["origin_group"], df["cut"]).to_string())

print(f"\nDone. Ready for analysis.")


ORIGIN: Recycled
  Band 0.25-0.5ct... 7 records | Running total: 7
  Band 0.51-0.75ct... 29 records | Running total: 36
  Band 0.76-1.0ct... 39 records | Running total: 75
  Band 1.01-1.5ct... 40 records | Running total: 115
  Band 1.51-2.0ct... 14 records | Running total: 129
  Band 2.01-3.0ct... 8 records | Running total: 137
  Band 3.01-5.0ct... 5 records | Running total: 142
  Band 5.01-20.0ct... 1 records | Running total: 143
ORIGIN DONE: 143 total for Recycled
Checkpoint saved: 143 total rows → diamonds_provenance_checkpoint.csv

ORIGIN: Canada
  Band 0.25-0.5ct... 2,777 records | Running total: 2,777
  Band 0.51-0.75ct... 2,699 records | Running total: 5,476
  Band 0.76-1.0ct... 1,526 records | Running total: 7,002
  Band 1.01-1.5ct... 1,897 records | Running total: 8,899
  Band 1.51-2.0ct... 701 records | Running total: 9,600
  Band 2.01-3.0ct... 485 records | Running total: 10,085
  Band 3.01-5.0ct... 82 records | Running total: 10,167
  Band 5.01-20.0ct... 15 records | Runni

KeyboardInterrupt: 

### 4b. Resume the natural-diamond collection
Reloads the saved Canada and Recycled checkpoint, re-collects **Angola** from the start, then collects **Botswana**. Duplicate listings are removed, prices and carat weights are converted to numbers, and the final natural-diamond file (`diamonds_provenance.csv`) is saved with a summary by origin.

In [16]:
import requests
import pandas as pd
import time
import random

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
    "Accept": "application/json, text/plain, */*",
    "Referer": "https://www.brilliantearth.com/diamond/shop-all/",
}

BASE = "https://www.brilliantearth.com/api/v1/plp/products/"

FIELDS = [
    "id", "price", "carat", "cut", "color", "clarity", "shape",
    "origin", "report", "polish", "symmetry", "fluorescence",
    "depth", "table", "length_width_ratio", "length",
]

CHECKPOINT_PATH = "diamonds_provenance_checkpoint.csv"
OUTPUT_PATH     = "diamonds_provenance.csv"

CARAT_BANDS = [
    (0.25, 0.50),
    (0.51, 0.75),
    (0.76, 1.00),
    (1.01, 1.50),
    (1.51, 2.00),
    (2.01, 3.00),
    (3.01, 5.00),
    (5.01, 20.0),
]

def scrape_single(origin_label, origin_param, carat_min=0.25, carat_max=20.0):
    records = []
    page = 1
    consecutive_errors = 0
    total_available = None
    seen_ids = set()

    while True:
        params = {
            "display": 50,
            "page": page,
            "currency": "USD",
            "product_class": "Loose Diamonds",
            "order_by": "most_popular",
            "order_method": "asc",
            "min_carat": carat_min,
            "max_carat": carat_max,
        }
        if origin_param:
            params["origins"] = origin_param

        try:
            r = requests.get(BASE, params=params, headers=headers, timeout=15)

            if r.status_code != 200:
                consecutive_errors += 1
                print(f"    Page {page}: HTTP {r.status_code} — waiting 15s")
                if consecutive_errors >= 5:
                    break
                time.sleep(15)
                continue

            data = r.json()
            products = (data.get("results") or
                        data.get("data", {}).get("products") or
                        data.get("products") or [])

            if total_available is None:
                total_available = int(data.get("total") or
                                      data.get("data", {}).get("total") or 0)

            if not products:
                break

            page_ids = {p.get("id") for p in products}
            if page_ids and page_ids.issubset(seen_ids):
                print(f"    Page {page}: duplicate loop — stopping")
                break
            seen_ids.update(page_ids)

            for p in products:
                rec = {field: p.get(field) for field in FIELDS}
                rec["origin_group"] = origin_label
                records.append(rec)

            consecutive_errors = 0

            if page * 50 >= total_available:
                break

            page += 1
            time.sleep(random.uniform(0.8, 1.8))

        except Exception as e:
            consecutive_errors += 1
            print(f"    Page {page}: Exception — {e}")
            if consecutive_errors >= 5:
                break
            time.sleep(10)

    return records


def scrape_banded(origin_label, origin_param, bands):
    print(f"\n{'='*60}")
    print(f"ORIGIN: {origin_label}")
    print(f"{'='*60}")
    all_records = []
    for carat_min, carat_max in bands:
        print(f"\n  Band {carat_min}-{carat_max}ct:")
        band_records = scrape_single(origin_label, origin_param, carat_min, carat_max)
        all_records.extend(band_records)
        print(f"  Band done: {len(band_records):,} | Running total: {len(all_records):,}")
        time.sleep(random.uniform(2, 4))
    print(f"DONE: {len(all_records):,} total for {origin_label}")
    return all_records


# ── LOAD CHECKPOINT ───────────────────────────────────────────────────────────
print("Loading checkpoint...")
existing = pd.read_csv(CHECKPOINT_PATH)
print(f"Checkpoint has {len(existing):,} rows")
print(existing["origin_group"].value_counts().to_string())

# Convert to list of dicts to extend with new data
all_data = existing.to_dict("records")

# ── RESUME FROM ANGOLA ────────────────────────────────────────────────────────
# Angola failed mid-scrape — rescrape it fully, deduplicate at the end
# Botswana Sort hasn't started yet

for label, param in [("Angola", "Angola"), ("Botswana Sort", "Botswana Sort")]:
    records = scrape_banded(label, param, CARAT_BANDS)
    all_data.extend(records)
    pd.DataFrame(all_data).to_csv(CHECKPOINT_PATH, index=False)
    print(f"Checkpoint saved: {len(all_data):,} total rows")
    time.sleep(random.uniform(5, 10))


# ── CLEAN & SAVE ──────────────────────────────────────────────────────────────
df = pd.DataFrame(all_data)

before = len(df)
df = df.drop_duplicates(subset=["id"])
print(f"\nDeduplication: {before:,} → {len(df):,} rows ({before - len(df):,} dupes removed)")

for col in ["price", "carat", "depth", "table", "length_width_ratio", "length"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

df = df.dropna(subset=["price", "carat"])
df.to_csv(OUTPUT_PATH, index=False)

# ── SUMMARY ───────────────────────────────────────────────────────────────────
print(f"\n{'='*60}")
print(f"FINAL DATASET SAVED: diamonds_provenance.csv")
print(f"{'='*60}")
print(f"Total rows: {len(df):,}")

print(f"\nCount by origin_group:")
print(df["origin_group"].value_counts().to_string())

print(f"\nMedian price by origin:")
print(df.groupby("origin_group")["price"].median().sort_values(ascending=False).to_string())

print(f"\nMedian carat by origin:")
print(df.groupby("origin_group")["carat"].median().sort_values(ascending=False).to_string())

print(f"\nCut distribution by origin:")
print(pd.crosstab(df["origin_group"], df["cut"]).to_string())

print(f"\nDone. Ready for analysis.")

Loading checkpoint...
Checkpoint has 10,325 rows
origin_group
Canada      10182
Recycled      143

ORIGIN: Angola

  Band 0.25-0.5ct:
  Band done: 4,596 | Running total: 4,596

  Band 0.51-0.75ct:
  Band done: 9,114 | Running total: 13,710

  Band 0.76-1.0ct:
  Band done: 4,982 | Running total: 18,692

  Band 1.01-1.5ct:
    Page 201: duplicate loop — stopping
  Band done: 10,000 | Running total: 28,692

  Band 1.51-2.0ct:
  Band done: 3,036 | Running total: 31,728

  Band 2.01-3.0ct:
  Band done: 2,469 | Running total: 34,197

  Band 3.01-5.0ct:
  Band done: 849 | Running total: 35,046

  Band 5.01-20.0ct:
  Band done: 178 | Running total: 35,224
DONE: 35,224 total for Angola
Checkpoint saved: 45,549 total rows

ORIGIN: Botswana Sort

  Band 0.25-0.5ct:
    Page 202: duplicate loop — stopping
  Band done: 10,050 | Running total: 10,050

  Band 0.51-0.75ct:
    Page 201: duplicate loop — stopping
  Band done: 10,000 | Running total: 20,050

  Band 0.76-1.0ct:
  Band done: 9,653 | Runni

## 5. Design carat-weight bands for lab-grown diamonds
Brilliant Earth's lab-grown inventory is much larger than its natural inventory, so the natural-diamond bands are too wide to page through. The next four cells count listings per band and split any oversized band into narrower ranges until all **27 bands** fit within the pagination limit.

In [18]:
print("="*60)
print("STEP 1: Band sizing check for Lab Created diamonds")
print("="*60)

CARAT_BANDS_LAB = [
    (0.25, 0.50),
    (0.51, 0.75),
    (0.76, 1.00),
    (1.01, 1.50),
    (1.51, 2.00),
    (2.01, 3.00),
    (3.01, 5.00),
    (5.01, 70.0),
]

total_lab = 0

for lo, hi in CARAT_BANDS_LAB:
    params = {
        "display": 1, "page": 1,
        "currency": "USD",
        "product_class": "Lab Created Colorless Diamonds",
        "min_carat": lo,
        "max_carat": hi,
    }
    r = requests.get(BASE, params=params, headers=headers, timeout=15)
    data = r.json()
    total = int(data.get("total") or
                data.get("data", {}).get("total") or 0)
    pages = -(-total // 50)
    ok = "✅" if pages <= 450 else "⚠️ TOO BIG"
    print(f"  {lo}-{hi}ct: {total:>8,} diamonds | {pages:>5} pages | {ok}")
    total_lab += total
    time.sleep(1.0)

print(f"\n  TOTAL Lab Created available: {total_lab:,}")

STEP 1: Band sizing check for Lab Created diamonds
  0.25-0.5ct:    9,314 diamonds |   187 pages | ✅
  0.51-0.75ct:   16,552 diamonds |   332 pages | ✅
  0.76-1.0ct:   16,742 diamonds |   335 pages | ✅
  1.01-1.5ct:   55,739 diamonds |  1115 pages | ⚠️ TOO BIG
  1.51-2.0ct:   52,227 diamonds |  1045 pages | ⚠️ TOO BIG
  2.01-3.0ct:   69,210 diamonds |  1385 pages | ⚠️ TOO BIG
  3.01-5.0ct:   61,539 diamonds |  1231 pages | ⚠️ TOO BIG
  5.01-70.0ct:   34,931 diamonds |   699 pages | ⚠️ TOO BIG

  TOTAL Lab Created available: 316,254


In [19]:
print("Finding sub-band split points for oversized bands...")
print()

oversized = [
    (1.01, 1.50),
    (1.51, 2.00),
    (2.01, 3.00),
    (3.01, 5.00),
    (5.01, 70.0),
]

# Test split points within each oversized band
split_candidates = {
    (1.01, 1.50): [(1.01,1.20),(1.21,1.35),(1.36,1.50)],
    (1.51, 2.00): [(1.51,1.70),(1.71,1.85),(1.86,2.00)],
    (2.01, 3.00): [(2.01,2.30),(2.31,2.60),(2.61,3.00)],
    (3.01, 5.00): [(3.01,3.50),(3.51,4.00),(4.01,5.00)],
    (5.01, 70.0): [(5.01,7.00),(7.01,10.0),(10.01,70.0)],
}

all_bands = []

# First add the bands that are already fine
ok_bands = [(0.25,0.50),(0.51,0.75),(0.76,1.00)]
for lo, hi in ok_bands:
    all_bands.append((lo, hi))
    print(f"  {lo}-{hi}ct: ✅ already fine")

print()

# Now test splits for oversized bands
for (band_lo, band_hi), splits in split_candidates.items():
    print(f"  Band {band_lo}-{band_hi}ct — testing splits:")
    for lo, hi in splits:
        params = {
            "display": 1, "page": 1,
            "currency": "USD",
            "product_class": "Lab Created Colorless Diamonds",
            "min_carat": lo,
            "max_carat": hi,
        }
        r = requests.get(BASE, params=params, headers=headers, timeout=15)
        data = r.json()
        total = int(data.get("total") or
                    data.get("data", {}).get("total") or 0)
        pages = -(-total // 50)
        ok = "✅" if pages <= 450 else "⚠️ STILL TOO BIG"
        print(f"    {lo}-{hi}ct: {total:>8,} diamonds | {pages:>5} pages | {ok}")
        if pages <= 450:
            all_bands.append((lo, hi))
        time.sleep(1.0)
    print()

print(f"\nFinal band list ({len(all_bands)} bands):")
for lo, hi in sorted(all_bands):
    print(f"  {lo}-{hi}ct")

Finding sub-band split points for oversized bands...

  0.25-0.5ct: ✅ already fine
  0.51-0.75ct: ✅ already fine
  0.76-1.0ct: ✅ already fine

  Band 1.01-1.5ct — testing splits:
    1.01-1.2ct:   31,752 diamonds |   636 pages | ⚠️ STILL TOO BIG
    1.21-1.35ct:   11,305 diamonds |   227 pages | ✅
    1.36-1.5ct:   12,682 diamonds |   254 pages | ✅

  Band 1.51-2.0ct — testing splits:
    1.51-1.7ct:   29,561 diamonds |   592 pages | ⚠️ STILL TOO BIG
    1.71-1.85ct:   10,834 diamonds |   217 pages | ✅
    1.86-2.0ct:   11,832 diamonds |   237 pages | ✅

  Band 2.01-3.0ct — testing splits:
    2.01-2.3ct:   35,112 diamonds |   703 pages | ⚠️ STILL TOO BIG
    2.31-2.6ct:   19,416 diamonds |   389 pages | ✅
    2.61-3.0ct:   14,681 diamonds |   294 pages | ✅

  Band 3.01-5.0ct — testing splits:
    3.01-3.5ct:   26,981 diamonds |   540 pages | ⚠️ STILL TOO BIG
    3.51-4.0ct:   12,836 diamonds |   257 pages | ✅
    4.01-5.0ct:   21,722 diamonds |   435 pages | ✅

  Band 5.01-70.0ct — te

In [20]:
print("Second round splits for remaining oversized bands...")
print()

second_round = {
    (1.01, 1.20): [(1.01,1.08),(1.09,1.14),(1.15,1.20)],
    (1.51, 1.70): [(1.51,1.58),(1.59,1.64),(1.65,1.70)],
    (2.01, 2.30): [(2.01,2.10),(2.11,2.20),(2.21,2.30)],
    (3.01, 3.50): [(3.01,3.15),(3.16,3.30),(3.31,3.50)],
}

final_bands = [
    (0.25, 0.50),
    (0.51, 0.75),
    (0.76, 1.00),
    (1.21, 1.35),
    (1.36, 1.50),
    (1.71, 1.85),
    (1.86, 2.00),
    (2.31, 2.60),
    (2.61, 3.00),
    (3.51, 4.00),
    (4.01, 5.00),
    (5.01, 7.00),
    (7.01, 10.00),
    (10.01, 70.0),
]

for (band_lo, band_hi), splits in second_round.items():
    print(f"  Band {band_lo}-{band_hi}ct — testing splits:")
    for lo, hi in splits:
        params = {
            "display": 1, "page": 1,
            "currency": "USD",
            "product_class": "Lab Created Colorless Diamonds",
            "min_carat": lo,
            "max_carat": hi,
        }
        r = requests.get(BASE, params=params, headers=headers, timeout=15)
        data = r.json()
        total = int(data.get("total") or
                    data.get("data", {}).get("total") or 0)
        pages = -(-total // 50)
        ok = "✅" if pages <= 450 else "⚠️ STILL TOO BIG"
        print(f"    {lo}-{hi}ct: {total:>8,} diamonds | {pages:>5} pages | {ok}")
        if pages <= 450:
            final_bands.append((lo, hi))
        time.sleep(1.0)
    print()

final_bands_sorted = sorted(final_bands)
print(f"\nFinal confirmed band list ({len(final_bands_sorted)} bands):")
total_coverage = 0
for lo, hi in final_bands_sorted:
    params = {
        "display": 1, "page": 1,
        "currency": "USD",
        "product_class": "Lab Created Colorless Diamonds",
        "min_carat": lo,
        "max_carat": hi,
    }
    r = requests.get(BASE, params=params, headers=headers, timeout=15)
    data = r.json()
    total = int(data.get("total") or
                data.get("data", {}).get("total") or 0)
    pages = -(-total // 50)
    ok = "✅" if pages <= 450 else "⚠️ TOO BIG"
    total_coverage += total
    print(f"  {lo}-{hi}ct: {total:>8,} | {pages:>4} pages | {ok}")
    time.sleep(0.8)

print(f"\nTotal diamonds across all bands: {total_coverage:,}")
print(f"Expected total: ~316,254")
print(f"Coverage: {total_coverage/316254*100:.1f}%")

Second round splits for remaining oversized bands...

  Band 1.01-1.2ct — testing splits:
    1.01-1.08ct:   21,852 diamonds |   438 pages | ✅
    1.09-1.14ct:    6,402 diamonds |   129 pages | ✅
    1.15-1.2ct:    3,498 diamonds |    70 pages | ✅

  Band 1.51-1.7ct — testing splits:
    1.51-1.58ct:   19,747 diamonds |   395 pages | ✅
    1.59-1.64ct:    6,017 diamonds |   121 pages | ✅
    1.65-1.7ct:    3,797 diamonds |    76 pages | ✅

  Band 2.01-2.3ct — testing splits:
    2.01-2.1ct:   24,458 diamonds |   490 pages | ⚠️ STILL TOO BIG
    2.11-2.2ct:    5,802 diamonds |   117 pages | ✅
    2.21-2.3ct:    4,852 diamonds |    98 pages | ✅

  Band 3.01-3.5ct — testing splits:
    3.01-3.15ct:   18,045 diamonds |   361 pages | ✅
    3.16-3.3ct:    4,450 diamonds |    89 pages | ✅
    3.31-3.5ct:    4,486 diamonds |    90 pages | ✅


Final confirmed band list (25 bands):
  0.25-0.5ct:    9,314 |  187 pages | ✅
  0.51-0.75ct:   16,552 |  332 pages | ✅
  0.76-1.0ct:   16,742 |  335 page

In [21]:
print("Final split for 2.01-2.1ct band...")
print()

last_splits = [(2.01, 2.05), (2.06, 2.10)]

confirmed_final_bands = [
    (0.25, 0.50),
    (0.51, 0.75),
    (0.76, 1.00),
    (1.01, 1.08),
    (1.09, 1.14),
    (1.15, 1.20),
    (1.21, 1.35),
    (1.36, 1.50),
    (1.51, 1.58),
    (1.59, 1.64),
    (1.65, 1.70),
    (1.71, 1.85),
    (1.86, 2.00),
    (2.11, 2.20),
    (2.21, 2.30),
    (2.31, 2.60),
    (2.61, 3.00),
    (3.01, 3.15),
    (3.16, 3.30),
    (3.31, 3.50),
    (3.51, 4.00),
    (4.01, 5.00),
    (5.01, 7.00),
    (7.01, 10.00),
    (10.01, 70.0),
]

for lo, hi in last_splits:
    params = {
        "display": 1, "page": 1,
        "currency": "USD",
        "product_class": "Lab Created Colorless Diamonds",
        "min_carat": lo,
        "max_carat": hi,
    }
    r = requests.get(BASE, params=params, headers=headers, timeout=15)
    data = r.json()
    total = int(data.get("total") or
                data.get("data", {}).get("total") or 0)
    pages = -(-total // 50)
    ok = "✅" if pages <= 450 else "⚠️ STILL TOO BIG"
    print(f"  {lo}-{hi}ct: {total:>8,} diamonds | {pages:>4} pages | {ok}")
    if pages <= 450:
        confirmed_final_bands.append((lo, hi))
    time.sleep(1.0)

confirmed_final_bands = sorted(confirmed_final_bands)

print(f"\nAll {len(confirmed_final_bands)} bands confirmed:")
grand_total = 0
all_green = True
for lo, hi in confirmed_final_bands:
    params = {
        "display": 1, "page": 1,
        "currency": "USD",
        "product_class": "Lab Created Colorless Diamonds",
        "min_carat": lo,
        "max_carat": hi,
    }
    r = requests.get(BASE, params=params, headers=headers, timeout=15)
    data = r.json()
    total = int(data.get("total") or
                data.get("data", {}).get("total") or 0)
    pages = -(-total // 50)
    ok = "✅" if pages <= 450 else "⚠️ TOO BIG"
    if pages > 450:
        all_green = False
    grand_total += total
    print(f"  {lo}-{hi}ct: {total:>8,} | {pages:>4} pages | {ok}")
    time.sleep(0.8)

print(f"\nGrand total: {grand_total:,}")
print(f"Coverage: {grand_total/316254*100:.1f}%")
print(f"\n{'✅ ALL BANDS GREEN — READY TO SCRAPE' if all_green else '⚠️ STILL HAVE OVERSIZED BANDS'}")

Final split for 2.01-2.1ct band...

  2.01-2.05ct:   14,105 diamonds |  283 pages | ✅
  2.06-2.1ct:   10,353 diamonds |  208 pages | ✅

All 27 bands confirmed:
  0.25-0.5ct:    9,314 |  187 pages | ✅
  0.51-0.75ct:   16,552 |  332 pages | ✅
  0.76-1.0ct:   16,742 |  335 pages | ✅
  1.01-1.08ct:   21,852 |  438 pages | ✅
  1.09-1.14ct:    6,402 |  129 pages | ✅
  1.15-1.2ct:    3,498 |   70 pages | ✅
  1.21-1.35ct:   11,305 |  227 pages | ✅
  1.36-1.5ct:   12,682 |  254 pages | ✅
  1.51-1.58ct:   19,747 |  395 pages | ✅
  1.59-1.64ct:    6,017 |  121 pages | ✅
  1.65-1.7ct:    3,797 |   76 pages | ✅
  1.71-1.85ct:   10,834 |  217 pages | ✅
  1.86-2.0ct:   11,832 |  237 pages | ✅
  2.01-2.05ct:   14,105 |  283 pages | ✅
  2.06-2.1ct:   10,353 |  208 pages | ✅
  2.11-2.2ct:    5,802 |  117 pages | ✅
  2.21-2.3ct:    4,852 |   98 pages | ✅
  2.31-2.6ct:   19,416 |  389 pages | ✅
  2.61-3.0ct:   14,681 |  294 pages | ✅
  3.01-3.15ct:   18,045 |  361 pages | ✅
  3.16-3.3ct:    4,450 |   89 p

## 6. Collect lab-grown diamonds
Collects lab-grown listings across the 27 bands. To keep the lab-grown sample a similar size to the natural sample (about 94,000), collection is capped at **3,500 diamonds per band**. Duplicates are removed and the result is saved as `diamonds_labgrown.csv`.

In [24]:
CARAT_BANDS_LAB = [
    (0.25, 0.50),
    (0.51, 0.75),
    (0.76, 1.00),
    (1.01, 1.08),
    (1.09, 1.14),
    (1.15, 1.20),
    (1.21, 1.35),
    (1.36, 1.50),
    (1.51, 1.58),
    (1.59, 1.64),
    (1.65, 1.70),
    (1.71, 1.85),
    (1.86, 2.00),
    (2.01, 2.05),
    (2.06, 2.10),
    (2.11, 2.20),
    (2.21, 2.30),
    (2.31, 2.60),
    (2.61, 3.00),
    (3.01, 3.15),
    (3.16, 3.30),
    (3.31, 3.50),
    (3.51, 4.00),
    (4.01, 5.00),
    (5.01, 7.00),
    (7.01, 10.00),
    (10.01, 70.0),
]

FIELDS = [
    "id", "price", "carat", "cut", "color", "clarity", "shape",
    "origin", "report", "polish", "symmetry", "fluorescence",
    "depth", "table", "length_width_ratio", "length",
]

CHECKPOINT_PATH = "diamonds_labgrown_checkpoint.csv"
OUTPUT_PATH     = "diamonds_labgrown.csv"

# Cap per band — 3,500 gives ~94,500 total across 27 bands
# This roughly matches the natural diamond dataset size (~94,448)
CAP_PER_BAND = 3500


def scrape_band_lab(carat_min, carat_max, cap=3500):
    records = []
    page = 1
    consecutive_errors = 0
    total_available = None
    seen_ids = set()

    while len(records) < cap:
        params = {
            "display": 50,
            "page": page,
            "currency": "USD",
            "product_class": "Lab Created Colorless Diamonds",
            "order_by": "most_popular",
            "order_method": "asc",
            "min_carat": carat_min,
            "max_carat": carat_max,
        }

        try:
            r = requests.get(BASE, params=params, headers=headers, timeout=15)

            if r.status_code != 200:
                consecutive_errors += 1
                print(f"    HTTP {r.status_code} on page {page} — waiting 15s")
                if consecutive_errors >= 5:
                    print(f"    5 errors — stopping band")
                    break
                time.sleep(15)
                continue

            data = r.json()
            products = (data.get("results") or
                        data.get("data", {}).get("products") or
                        data.get("products") or [])

            if total_available is None:
                total_available = int(data.get("total") or
                                      data.get("data", {}).get("total") or 0)

            if not products:
                break

            page_ids = {p.get("id") for p in products}
            if page_ids and page_ids.issubset(seen_ids):
                print(f"    Page {page}: duplicate loop — stopping")
                break
            seen_ids.update(page_ids)

            for p in products:
                if len(records) >= cap:
                    break
                rec = {field: p.get(field) for field in FIELDS}
                rec["diamond_type"] = "Lab Grown"
                records.append(rec)

            consecutive_errors = 0

            # Stop when the cap or the last page is reached
            if len(records) >= cap:
                print(f"    Hit cap of {cap:,}")
                break
            if page * 50 >= total_available:
                print(f"    Reached last page")
                break

            page += 1
            time.sleep(random.uniform(0.8, 1.5))

        except Exception as e:
            consecutive_errors += 1
            print(f"    Page {page}: Exception — {e}")
            if consecutive_errors >= 5:
                break
            time.sleep(10)

    return records, total_available or 0


# ── MAIN SCRAPE ───────────────────────────────────────────────────────────
print("="*60)
print("LAB GROWN DIAMOND SCRAPE — CAPPED VERSION")
print(f"27 bands | ~{CAP_PER_BAND * len(CARAT_BANDS_LAB):,} target diamonds")
print(f"Cap per band: {CAP_PER_BAND:,}")
print(f"Est. runtime: 30-40 minutes")
print("="*60)

all_data = []
grand_total = 0

for band_num, (lo, hi) in enumerate(CARAT_BANDS_LAB, 1):
    print(f"\n[{band_num:>2}/{len(CARAT_BANDS_LAB)}] "
          f"Band {lo}-{hi}ct...", end=" ", flush=True)

    band_records, band_total = scrape_band_lab(lo, hi, cap=CAP_PER_BAND)
    all_data.extend(band_records)
    grand_total += len(band_records)

    print(f"{len(band_records):,} collected "
          f"(of {band_total:,} available) | "
          f"Running total: {grand_total:,}")

    # Checkpoint after every band
    pd.DataFrame(all_data).to_csv(CHECKPOINT_PATH, index=False)

    time.sleep(random.uniform(2, 3))


# ── CLEAN & SAVE ──────────────────────────────────────────────────────────
print("\n\nCleaning dataset...")

df_lab = pd.DataFrame(all_data)

before = len(df_lab)
df_lab = df_lab.drop_duplicates(subset=["id"])
print(f"Deduplication: {before:,} → {len(df_lab):,} rows")

for col in ["price", "carat", "depth", "table", "length_width_ratio"]:
    df_lab[col] = pd.to_numeric(df_lab[col], errors="coerce")

df_lab = df_lab.dropna(subset=["price", "carat"])
df_lab.to_csv(OUTPUT_PATH, index=False)

# ── SUMMARY ───────────────────────────────────────────────────────────────
print(f"\n{'='*60}")
print(f"FINAL DATASET: diamonds_labgrown.csv")
print(f"{'='*60}")
print(f"Total rows: {len(df_lab):,}")
print(f"\nPrice range:  ${df_lab['price'].min():,.0f} — "
      f"${df_lab['price'].max():,.0f}")
print(f"Carat range:  {df_lab['carat'].min():.2f} — "
      f"{df_lab['carat'].max():.2f}ct")
print(f"\nCut distribution:")
print(df_lab["cut"].value_counts().to_string())
print(f"\nColor distribution:")
print(df_lab["color"].value_counts().to_string())
print(f"\nClarity distribution:")
print(df_lab["clarity"].value_counts().to_string())
print(f"\nReport distribution:")
print(df_lab["report"].value_counts().to_string())
print(f"\nMedian price:  ${df_lab['price'].median():,.0f}")
print(f"Median carat:  {df_lab['carat'].median():.2f}ct")
print(f"\nDone. Lab grown dataset ready.")

LAB GROWN DIAMOND SCRAPE — CAPPED VERSION
27 bands | ~94,500 target diamonds
Cap per band: 3,500
Est. runtime: 30-40 minutes

[ 1/27] Band 0.25-0.5ct...     Hit cap of 3,500
3,500 collected (of 9,314 available) | Running total: 3,500

[ 2/27] Band 0.51-0.75ct...     Hit cap of 3,500
3,500 collected (of 16,552 available) | Running total: 7,000

[ 3/27] Band 0.76-1.0ct...     Hit cap of 3,500
3,500 collected (of 16,742 available) | Running total: 10,500

[ 4/27] Band 1.01-1.08ct...     Hit cap of 3,500
3,500 collected (of 21,852 available) | Running total: 14,000

[ 5/27] Band 1.09-1.14ct...     Hit cap of 3,500
3,500 collected (of 6,402 available) | Running total: 17,500

[ 6/27] Band 1.15-1.2ct...     Reached last page
3,498 collected (of 3,498 available) | Running total: 20,998

[ 7/27] Band 1.21-1.35ct...     Hit cap of 3,500
3,500 collected (of 11,305 available) | Running total: 24,498

[ 8/27] Band 1.36-1.5ct...     Hit cap of 3,500
3,500 collected (of 12,682 available) | Running t

## 7. Combine natural and lab-grown diamonds
Loads both files, labels each diamond as Natural or Lab Grown, and compares the quality distributions of the two groups.

In [14]:
import pandas as pd
import numpy as np

# ── LOAD BOTH DATASETS ────────────────────────────────────────
print("Loading datasets...")
df_nat = pd.read_csv("diamonds_provenance.csv")
df_lab = pd.read_csv("diamonds_labgrown.csv")

# Clean numerics
for col in ["price","carat","depth","table","length_width_ratio"]:
    df_nat[col] = pd.to_numeric(df_nat[col], errors="coerce")
    df_lab[col] = pd.to_numeric(df_lab[col], errors="coerce")

df_nat = df_nat.dropna(subset=["price","carat"])
df_lab = df_lab.dropna(subset=["price","carat"])

# Add type labels
df_nat["diamond_type"] = "Natural"
df_lab["diamond_type"] = "Lab Grown"

print(f"Natural:   {len(df_nat):,} diamonds")
print(f"Lab Grown: {len(df_lab):,} diamonds")

# ── QUALITY DISTRIBUTION COMPARISON ──────────────────────────
print("\n" + "="*60)
print("QUALITY DISTRIBUTION COMPARISON")
print("="*60)

cut_order     = {"Fair":1,"Good":2,"Very Good":3,"Ideal":4,"Super Ideal":5}
color_order   = {"K":1,"J":2,"I":3,"H":4,"G":5,"F":6,"E":7,"D":8}
clarity_order = {"SI2":1,"SI1":2,"VS2":3,"VS1":4,
                 "VVS2":5,"VVS1":6,"IF":7,"FL":8}

for df in [df_nat, df_lab]:
    df["cut_n"]     = df["cut"].map(cut_order)
    df["color_n"]   = df["color"].map(color_order)
    df["clarity_n"] = df["clarity"].map(clarity_order)
    df["log_price"] = np.log(df["price"])
    df["log_carat"] = np.log(df["carat"])

print(f"\nCut distribution (%):")
cut_pct = pd.crosstab(df_nat["cut"].fillna("Unknown"),
                       pd.Series(["Natural"]*len(df_nat)),
                       normalize="columns").mul(100)
lab_cut = pd.crosstab(df_lab["cut"].fillna("Unknown"),
                       pd.Series(["Lab Grown"]*len(df_lab)),
                       normalize="columns").mul(100)
cut_compare = pd.concat([cut_pct, lab_cut], axis=1)
cut_compare.columns = ["Natural %", "Lab Grown %"]
print(cut_compare.round(1).to_string())

print(f"\nColor distribution (%):")
nat_color = df_nat["color"].value_counts(normalize=True).mul(100).round(1)
lab_color = df_lab["color"].value_counts(normalize=True).mul(100).round(1)
color_compare = pd.DataFrame({"Natural %": nat_color,
                               "Lab Grown %": lab_color})
print(color_compare.to_string())

print(f"\nClarity distribution (%):")
nat_clar = df_nat["clarity"].value_counts(normalize=True).mul(100).round(1)
lab_clar = df_lab["clarity"].value_counts(normalize=True).mul(100).round(1)
clar_compare = pd.DataFrame({"Natural %": nat_clar,
                              "Lab Grown %": lab_clar})
print(clar_compare.to_string())

print(f"\nCertifier distribution (%):")
nat_rep = df_nat["report"].value_counts(normalize=True).mul(100).round(1)
lab_rep = df_lab["report"].value_counts(normalize=True).mul(100).round(1)
rep_compare = pd.DataFrame({"Natural %": nat_rep,
                             "Lab Grown %": lab_rep})
print(rep_compare.to_string())

# ── CARAT DISTRIBUTION ────────────────────────────────────────
print(f"\n" + "="*60)
print("CARAT DISTRIBUTION COMPARISON")
print("="*60)

carat_bands = [(0.25,0.50,"0.25-0.50"),
               (0.51,0.75,"0.51-0.75"),
               (0.76,1.00,"0.76-1.00"),
               (1.01,1.50,"1.01-1.50"),
               (1.51,2.00,"1.51-2.00"),
               (2.01,3.00,"2.01-3.00"),
               (3.01,5.00,"3.01-5.00"),
               (5.01,30.0,"5.01+    ")]

print(f"\n{'Band':<12} {'Natural %':>10} {'Lab Grown %':>12} "
      f"{'Nat Median $':>14} {'Lab Median $':>13}")
print("-"*65)

for lo, hi, label in carat_bands:
    nat_mask = (df_nat["carat"]>=lo) & (df_nat["carat"]<=hi)
    lab_mask = (df_lab["carat"]>=lo) & (df_lab["carat"]<=hi)
    nat_pct  = nat_mask.sum() / len(df_nat) * 100
    lab_pct  = lab_mask.sum() / len(df_lab) * 100
    nat_med  = df_nat[nat_mask]["price"].median() if nat_mask.sum()>0 else np.nan
    lab_med  = df_lab[lab_mask]["price"].median() if lab_mask.sum()>0 else np.nan
    print(f"{label:<12} {nat_pct:>9.1f}% {lab_pct:>11.1f}%  "
          f"${nat_med:>12,.0f}  ${lab_med:>11,.0f}")

# ── PRICE COMPARISON AT MATCHED QUALITY ───────────────────────
print(f"\n" + "="*60)
print("PRICE COMPARISON AT MATCHED QUALITY TIERS")
print("Same cut + color + clarity + carat band")
print("="*60)

df_nat["carat_band"] = pd.cut(df_nat["carat"],
    bins=[0,0.5,0.75,1.0,1.5,2.0,3.0,5.0,30],
    labels=["0.25-0.5","0.51-0.75","0.76-1.0","1.01-1.5",
            "1.51-2.0","2.01-3.0","3.01-5.0","5.01+"])

df_lab["carat_band"] = pd.cut(df_lab["carat"],
    bins=[0,0.5,0.75,1.0,1.5,2.0,3.0,5.0,30],
    labels=["0.25-0.5","0.51-0.75","0.76-1.0","1.01-1.5",
            "1.51-2.0","2.01-3.0","3.01-5.0","5.01+"])

# Median price per carat by type and carat band
nat_ppc = df_nat.groupby("carat_band")["price"].apply(
    lambda x: (x / df_nat.loc[x.index,"carat"]).median()).round(0)
lab_ppc = df_lab.groupby("carat_band")["price"].apply(
    lambda x: (x / df_lab.loc[x.index,"carat"]).median()).round(0)

ppc_compare = pd.DataFrame({
    "Natural $/ct":  nat_ppc,
    "Lab Grown $/ct": lab_ppc,
})
ppc_compare["Natural Premium"] = (
    (ppc_compare["Natural $/ct"] / ppc_compare["Lab Grown $/ct"]) - 1
) * 100

print(f"\nMedian price per carat by size band:")
print(f"{'Band':<12} {'Natural $/ct':>14} {'Lab $/ct':>10} "
      f"{'Natural Premium':>17}")
print("-"*58)
for band, row in ppc_compare.iterrows():
    print(f"{str(band):<12} ${row['Natural $/ct']:>12,.0f}  "
          f"${row['Lab Grown $/ct']:>8,.0f}  "
          f"{row['Natural Premium']:>+15.1f}%")

# ── BY ORIGIN VS LAB GROWN ────────────────────────────────────
print(f"\n" + "="*60)
print("NATURAL ORIGIN vs LAB GROWN — MEDIAN PRICE PER CARAT")
print("="*60)

print(f"\n{'Band':<12} {'Botswana':>10} {'Angola':>10} "
      f"{'Canada':>10} {'Lab Grown':>11}")
print("-"*55)

for lo, hi, label in carat_bands:
    lab_mask = (df_lab["carat"]>=lo) & (df_lab["carat"]<=hi)
    lab_med  = (df_lab[lab_mask]["price"] /
                df_lab[lab_mask]["carat"]).median() if lab_mask.sum()>0 else np.nan

    row = f"{label:<12}"
    for origin in ["Botswana Sort","Angola","Canada"]:
        nat_mask = ((df_nat["carat"]>=lo) &
                    (df_nat["carat"]<=hi) &
                    (df_nat["origin_group"]==origin))
        med = (df_nat[nat_mask]["price"] /
               df_nat[nat_mask]["carat"]).median() if nat_mask.sum()>0 else np.nan
        row += f" ${med:>8,.0f}" if not np.isnan(med) else f" {'N/A':>9}"
    row += f"  ${lab_med:>8,.0f}" if not np.isnan(lab_med) else f"  {'N/A':>9}"
    print(row)

print(f"\nNote: Raw price/carat comparison — not quality controlled")
print(f"Regression analysis required for fair comparison")
print(f"\nDatasets loaded and ready for merged analysis.")

Loading datasets...
Natural:   94,448 diamonds
Lab Grown: 93,131 diamonds

QUALITY DISTRIBUTION COMPARISON

Cut distribution (%):
             Natural %  Lab Grown %
cut                                
Fair               0.2          0.1
Good               2.9          1.0
Ideal             39.0         34.9
Super Ideal       44.1         59.6
Very Good         13.7          4.4

Color distribution (%):
       Natural %  Lab Grown %
color                        
D           14.9         18.4
E           13.5         23.7
F           14.0         22.1
G           16.2         18.2
H           15.7          7.7
I           13.0          4.7
J            8.6          5.3
K            4.2          NaN

Clarity distribution (%):
         Natural %  Lab Grown %
clarity                        
FL             0.2          0.1
IF             4.6          0.7
SI1           16.5          9.1
SI2           17.0          1.9
VS1           16.7         36.3
VS2           17.6         23.3
VVS1      

Merges the two datasets into a single table, labeling lab-grown diamonds with their own origin group, and saves the final analysis file, `diamonds_merged_complete.csv` (187,579 rows).

In [15]:
import pandas as pd
import numpy as np

# ── MERGE DATASETS ────────────────────────────────────────────
print("Merging datasets...")

# Add origin_group to lab grown
df_lab["origin_group"] = "Lab Grown"

# Ensure both have the same columns
# Find common columns
nat_cols = set(df_nat.columns)
lab_cols = set(df_lab.columns)
common_cols = nat_cols.intersection(lab_cols)

print(f"\nNatural columns:   {len(nat_cols)}")
print(f"Lab Grown columns: {len(lab_cols)}")
print(f"Common columns:    {len(common_cols)}")

# Columns only in natural (origin-specific)
nat_only = nat_cols - lab_cols
lab_only = lab_cols - nat_cols
print(f"\nOnly in natural:   {nat_only}")
print(f"Only in lab grown: {lab_only}")

# Select columns to keep in merged dataset
# Keep all meaningful columns, fill missing with NaN
keep_cols = [
    "id", "price", "carat", "cut", "color", "clarity", "shape",
    "origin", "report", "polish", "symmetry", "fluorescence",
    "depth", "table", "length_width_ratio", "length",
    "diamond_type", "origin_group"
]

# Build merged dataframe
df_nat_clean = df_nat.copy()
df_lab_clean = df_lab.copy()

# Add missing columns as NaN so concat works cleanly
for col in keep_cols:
    if col not in df_nat_clean.columns:
        df_nat_clean[col] = np.nan
    if col not in df_lab_clean.columns:
        df_lab_clean[col] = np.nan

# Select only keep_cols from each
df_nat_merge = df_nat_clean[keep_cols].copy()
df_lab_merge = df_lab_clean[keep_cols].copy()

# Concatenate
df_merged = pd.concat([df_nat_merge, df_lab_merge], ignore_index=True)

# ── CLEAN TYPES ───────────────────────────────────────────────
for col in ["price", "carat", "depth", "table", "length_width_ratio"]:
    df_merged[col] = pd.to_numeric(df_merged[col], errors="coerce")

df_merged = df_merged.dropna(subset=["price","carat"])

# ── ADD DERIVED COLUMNS USEFUL FOR JMP ────────────────────────
# Log transforms
df_merged["log_price"] = np.log(df_merged["price"])
df_merged["log_carat"] = np.log(df_merged["carat"])

# Ordinal encodings
cut_order     = {"Fair":1,"Good":2,"Very Good":3,"Ideal":4,"Super Ideal":5}
color_order   = {"K":1,"J":2,"I":3,"H":4,"G":5,"F":6,"E":7,"D":8}
clarity_order = {"SI2":1,"SI1":2,"VS2":3,"VS1":4,
                 "VVS2":5,"VVS1":6,"IF":7,"FL":8}

df_merged["cut_n"]     = df_merged["cut"].map(cut_order)
df_merged["color_n"]   = df_merged["color"].map(color_order)
df_merged["clarity_n"] = df_merged["clarity"].map(clarity_order)

# Price per carat
df_merged["price_per_carat"] = df_merged["price"] / df_merged["carat"]

# Carat band
df_merged["carat_band"] = pd.cut(
    df_merged["carat"],
    bins=[0, 0.5, 0.75, 1.0, 1.5, 2.0, 3.0, 5.0, 100],
    labels=["0.25-0.5","0.51-0.75","0.76-1.0","1.01-1.5",
            "1.51-2.0","2.01-3.0","3.01-5.0","5.01+"]
)

# Natural vs lab binary flag (useful for JMP logistic regression)
df_merged["is_natural"] = (df_merged["diamond_type"] == "Natural").astype(int)

# ── SAVE ──────────────────────────────────────────────────────
output_path = "diamonds_merged_complete.csv"
df_merged.to_csv(output_path, index=False)

# ── SUMMARY ───────────────────────────────────────────────────
print(f"\n{'='*60}")
print(f"MERGED DATASET SAVED: diamonds_merged_complete.csv")
print(f"{'='*60}")
print(f"Total rows:    {len(df_merged):,}")
print(f"Total columns: {len(df_merged.columns)}")
print(f"\nColumns included:")
for col in df_merged.columns:
    print(f"  {col}")

print(f"\nRow counts by type:")
print(df_merged["diamond_type"].value_counts().to_string())

print(f"\nRow counts by origin_group:")
print(df_merged["origin_group"].value_counts().to_string())

print(f"\nPrice range:  ${df_merged['price'].min():,.0f} — "
      f"${df_merged['price'].max():,.0f}")
print(f"Carat range:  {df_merged['carat'].min():.2f} — "
      f"{df_merged['carat'].max():.2f}ct")

print(f"\nMissing values per column:")
missing = df_merged.isnull().sum()
missing = missing[missing > 0]
print(missing.to_string())

print(f"\nSample rows (one per origin_group):")
sample = df_merged.groupby("origin_group").first()
print(sample[["price","carat","cut","color","clarity",
              "diamond_type","log_price","log_carat"]].to_string())

print(f"\n✅ Ready for JMP analysis.")
print(f"File location: same folder as your notebook")

Merging datasets...

Natural columns:   24
Lab Grown columns: 24
Common columns:    24

Only in natural:   set()
Only in lab grown: set()

MERGED DATASET SAVED: diamonds_merged_complete.csv
Total rows:    187,579
Total columns: 26

Columns included:
  id
  price
  carat
  cut
  color
  clarity
  shape
  origin
  report
  polish
  symmetry
  fluorescence
  depth
  table
  length_width_ratio
  length
  diamond_type
  origin_group
  log_price
  log_carat
  cut_n
  color_n
  clarity_n
  price_per_carat
  carat_band
  is_natural

Row counts by type:
diamond_type
Natural      94448
Lab Grown    93131

Row counts by origin_group:
origin_group
Lab Grown        93131
Botswana Sort    49130
Angola           35009
Canada           10166
Recycled           143

Price range:  $180 — $646,510
Carat range:  0.25 — 25.84ct

Missing values per column:
fluorescence    159610

Sample rows (one per origin_group):
               price  carat          cut color clarity diamond_type  log_price  log_carat
ori